# 课程 5 作业 · token 账与 embedding 的边界

共 **100 分：简单题 10 分，两道中等题各 30 分，开放思考题 30 分**。只考主课讲过的内容；选做 Notebook 不是前置。

所有代码都在本地运行，没有付费模型调用。分词器使用课堂的 **Qwen/Qwen3-0.6B**，固定 revision `c1899de289a04d12100db370d81485cdf75e47ca`，通过 `tokenizers==0.22.2` 加载，不加载模型权重。题中的价格、窗口和模型配置凡标“假设”的，都不是任何厂商的真实数据。

选择已有的 **llm-from-zero** 内核，运行下面的准备单元。它会打印实际 Python 路径，便于确认没有选到其他环境。可从项目根目录或本课目录启动。

先按课堂说明准备好本地分词器文件；`load_tokenizer()` 只加载本地文件。若缺少文件或依赖，请回到本课 README 的准备步骤，不需要填写 API key。


In [ ]:
from pathlib import Path
import sys
import torch
import torch.nn.functional as F

candidates = [Path.cwd() / "05-token-embedding", Path.cwd()]
COURSE_DIR = next((p for p in candidates if (p / "tokenizer_tools.py").exists()), None)
if COURSE_DIR is None:
    raise RuntimeError("请从项目根目录或 05-token-embedding 目录运行。")
if str(COURSE_DIR) not in sys.path:
    sys.path.insert(0, str(COURSE_DIR))
from tokenizer_tools import load_tokenizer

tokenizer = load_tokenizer()
print("Python:", sys.executable)
print("词表大小:", tokenizer.get_vocab_size())


## 1 · 分清字符、token、ID 和向量（简单，10 分）

先运行下面的小实验。判断还原是否成功，要看对**完整 ID 序列**解码后的文本，不能只靠逐个 token 的显示效果。

In [ ]:
text = "星港🌙 2048"
encoded = tokenizer.encode(text, add_special_tokens=False)
print("原文:", text)
print("Python 字符数:", len(text))
print("token 数:", len(encoded.ids))
print("token 显示形式:", encoded.tokens)
print("token IDs:", encoded.ids)
print("完整序列还原:", tokenizer.decode(encoded.ids, skip_special_tokens=False))


**1a（3 分）** 区分三个对象，不要只抄输出。

- 字符数与 token 数是否必须相同？结合本次结果说明：____
- ID 表示“这个 token 在当前句子的第几个位置”，还是“它在词表中的编号”？____
- 本次完整解码是否等于原文？你检查的是哪两个对象？____

**1b（3 分）** Qwen3-0.6B 的输入 embedding 表形状是 `(151936, 1024)`。

| 问题 | 我的答案 |
|---|---|
| 一段输入有 50 个 token，查表输出的形状 | ____ |
| 输入变成 100 个 token，表的形状变不变 | ____ |
| 输入变成 100 个 token，输出的哪一维变了 | ____ |

**1c（4 分）** “两个 token ID 数字接近，它们对应的 embedding 就一定语义接近。”这句话成立吗？分别说明 ID 编号、表里的数值能告诉我们什么。

我的判断：____

我的解释：____

<details>
<summary>提示 1：先确认每个对象代表什么</summary>

字符、分词结果、词表编号和查表所得向量是四种对象。先说对象的含义，再讨论数量或形状，避免把编号当成位置。
</details>

<details>
<summary>提示 2：哪个量跟着输入走</summary>

表的行数由词表决定，每行几个数由模型决定；输出为每个输入位置保留一行。
</details>

## 2 · 上下文预算检查（中等，30 分）

你要把候选材料放进一段固定提示，检查最终请求能否放进上下文窗口。程序实际编码的是：

`prefix + 候选文本 + suffix`

窗口要同时装下输入和回答，所以还要给输出预留 `reserve_output` 个 token。**必须在最终拼好的字符串上调用一次 encode。** 不要把三部分分别编码再相加；拼接边界可能改变分词结果。

**2a（16 分）** 完成 `inspect_context_budget(tokenizer, prefix, candidates, suffix, window, reserve_output)`。返回与候选顺序一致的列表，每项包含：

| 键 | 内容 |
|---|---|
| `candidate` | 原候选字符串 |
| `input_tokens` | 最终字符串的 token 数，使用 `add_special_tokens=False` |
| `fits` | `input_tokens + reserve_output` 是否不超过 `window`；刚好相等也算放得下 |

假设输入类型正确，`window` 和 `reserve_output` 为非负整数；候选列表为空时返回空列表。这里只检查提供的文本，不包括聊天模板和其他消息，也不代表任一厂商的账单。

可用写法：

```python
"".join(["前缀", "正文", "后缀"])
for item in items:
    print(item)
encoding = tokenizer.encode("待测文本", add_special_tokens=False)
ids = encoding.ids
```

In [ ]:
def inspect_context_budget(tokenizer, prefix, candidates, suffix, window, reserve_output):
    # TODO 2a：完成整个函数；每个候选都在拼接后重新编码。
    pass

prefix = "任务：根据材料回答客户问题。\n<材料>\n"
suffix = "\n</材料>\n回答："
candidate_texts = [
    "退货需在签收后七天内申请。",
    "Returns must be requested within 7 days of delivery. 退货需在签收后七天内申请。",
    "商品未拆封且配件齐全时，可以申请无理由退货。" * 20,
]

budget_results = inspect_context_budget(tokenizer, prefix, candidate_texts, suffix, window=120, reserve_output=40)
print(budget_results)

**2b（6 分）** 用断言检查函数，至少覆盖：

1. 条数和候选顺序正确，没有漏掉放不下的候选。（2 分）
2. 选一个候选，单独编码其完整最终文本得到 `n`：`window = n + reserve_output` 时放得下，少 1 时放不下。（2 分）
3. 空候选列表返回空列表。（2 分）

`assert 条件` 在条件不成立时报错。边界里的 `n` 要独立测出来，不要直接拿同一个函数的返回值当标准答案。

In [ ]:
# TODO 2b：自己写断言，覆盖条数/顺序、刚好够/差 1 个 token、空列表。

**2c（8 分）** 选一个候选，分别打印：①完整字符串的 token 数；②三部分各自编码的 token 数之和。再用完整 ID 序列解码，检查是否还原最终字符串。计数相等也如实记录，不需要制造反例。

In [ ]:
# TODO 2c：测量一个候选的完整计数、分段计数之和，并检查完整解码。

- 我选择的候选与两种计数（3 分）：____
- 完整解码是否还原最终字符串，依据是什么（2 分）：____
- 即使这个例子两种计数相等，为什么还不能把“分段相加”写成通用预算算法（3 分）：____

<details>
<summary>提示 1：预算限制的究竟是哪一段文字？</summary>

先画出最终准备发送的文本范围。encode 的输入是否恰好是这个范围？再想一想输出放在哪里。
</details>

<details>
<summary>提示 2：把循环里的职责分开</summary>

每次循环先组装字符串，再编码计数，然后加上预留输出和窗口比较，最后把三个字段加入列表。测试时从完整字符串独立测出边界数量。
</details>

## 3 · 新材料的估算（中等，30 分）

一家跨境电商想做客服问答助手，材料有两类：英文产品手册和中文客服 FAQ。下面给了两段样本，来自题目编写，不是真实客户材料。

**3a（8 分）** 写代码测出两段样本各自的每字符 token 数（可以参考主课任务 2 的写法，但要在这里重新写出来并运行）。

In [ ]:
manual_sample = "To reset the device, press and hold the power button for ten seconds until the indicator light blinks blue. Then release the button and wait for the device to restart."
faq_sample = "问：订单发货后多久能到？答：国内一般三到五个工作日送达，偏远地区可能需要七个工作日。"

# TODO 3a：打印两段样本的字符数、token 数和每字符 token 数。

- 英文手册比例：____；中文 FAQ 比例：____。

**3b（14 分）** 用 3a 的比例估算两种方案。**以下都是题目假设：** 手册 120 页、每页约 3,000 个字符；FAQ 500 条、每条约 120 个字；系统提示等开销 300 tokens；每次回答预留并按用满计 800 tokens；窗口 131,072 tokens；价格输入 2 元/百万 tokens、输出 8 元/百万 tokens；每天 1,000 次问答。

- **方案甲**：每次问答都把手册和 FAQ 全部放进请求。
- **方案乙**：每次只找回最相关的 5 段材料，每段约 400 tokens，再放进请求。

| 项目 | 方案甲 | 方案乙 |
|---|---|---|
| 公式 | ____ | ____ |
| 单次输入 tokens | ____ | ____ |
| 加预留输出后能否放进窗口 | ____ | ____ |
| 单次费用（元） | ____ | ____ |
| 每天费用（元） | ____ | ____ |

未计入的项目（至少三项，其中一项要和方案乙有关）：____

方案乙省钱的前提是什么？它可能引入什么新风险？____

In [ ]:
# 可用 Python 辅助 3b 的计算。

**3c（8 分）** 一个**教学假设模型**：词表 V=64,000，维度 D=2,048，输入表与输出层**不共用**，总参数 1.5B。

- embedding 相关参数量和占总参数的比例：____
- 为支持更多语言，把词表扩到 128,000，其他不变：参数量增加多少，新的占比是多少，按 bfloat16（2 字节）多占多少 GB（10⁹ 字节）？____
- 扩词表之后，同一段中文的 token 数可能怎样变？这会怎样影响 3b 那种费用估算？只写推测和理由，不要求实验：____

<details>
<summary>提示 1：单位</summary>

单价先换成“元 / token”。百万是 10⁶。最后检查数量级：全部放进请求和只放几段，单次输入相差几十倍。
</details>

<details>
<summary>提示 2：共用与不共用</summary>

不共用就是输入表一张、输出层一张，各 `V × D`。占比 = embedding 参数 ÷ 总参数；扩词表后总参数也要跟着加。
</details>

## 4 · 面试追问：这些数字能证明什么？（开放思考，30 分）

面试官提出三个判断，请分别回应，不要求写新代码。

**4a（8 分）** “两个 token ID 相差很小，所以它们语义相近；ID 差值可以直接当作语义距离。”

我的判断：____

我的解释（区分编号与距离）：____

**4b（8 分）** “RAG 要的文本向量很简单：拿模型里的 embedding 表，把一句话每个 token 的向量取平均就行。”

我的判断：____

我的解释（结合主课 C 部分的实验，说明这样算出的向量缺什么，文本向量模型又多做了什么）：____

**4c（8 分）** “我们已经用 Qwen 分词器数好了正文的 token，直接乘单价，就是 DeepSeek 这次请求的真实账单。”

我的判断：____

我的解释（分词器、消息格式与请求范围、输出、实际用量和计费依据分别是什么）：____

**4d（6 分）** 从主课做过的操作中，提出一个小实验，检验上面任意一项说法。可以用拼接后分词、密度测量、消息格式拼接、小表查表、取平均等操作。说明观察到什么才支持你的结论，以及这个实验仍不能证明什么。只提出实验，不要求联网或付费调用。

我选择检验的说法：____

实验输入与只改变的一个条件：____

我要记录的结果，以及如何据此判断：____

仍不能证明的事：____

<details>
<summary>提示 1：区分计算结果和证据</summary>

一个数字、向量或费用公式能被正确算出来，不等于它能用于语义比较，也不等于它来自实际服务。不要用“算出来了”代替“已经验证用途”。
</details>

<details>
<summary>提示 2：为每项结论划定范围</summary>

逐项写清对象来自哪里、程序实际做了什么、没有获得什么证据。设计实验时只改变一个条件，保留原结果作比较。
</details>